# Corporate Intelligence Agent Demo

This notebook demonstrates the `Corporate Intelligence Agent` running from the modular Python files.

In [2]:
import os
import logging
from dotenv import load_dotenv
from google.adk.runners import InMemoryRunner
from agents import manager_agent

# Setup logging
logging.basicConfig(level=logging.INFO)

# Load API Key
load_dotenv()

if "GOOGLE_API_KEY" not in os.environ:
    print("⚠️ GOOGLE_API_KEY not found in environment. Please add it to your .env file.")
else:
    print("✅ API Key loaded successfully.")

✅ API Key loaded successfully.


In [36]:
import numpy as np
import yfinance as yf
import logging


# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_fundamentals(ticker: str) -> str:
    """Fetches financial fundamentals for a given ticker using yfinance."""
    logger.info(f"get_fundamentals called for: {ticker}")
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        # Extract key metrics with error handling
        revenue_growth = info.get('revenueGrowth', 'N/A')
        operating_margins = info.get('operatingMargins', 'N/A')
        total_cash = info.get('totalCash', 'N/A')
        total_debt = info.get('totalDebt', 'N/A')
        market_cap = info.get('marketCap', 'N/A')
        pe_ratio = info.get('trailingPE', 'N/A')
        
        # Format the data
        data = (
            f"Market Cap: {market_cap}\n"
            f"Revenue Growth: {revenue_growth}\n"
            f"Operating Margins: {operating_margins}\n"
            f"Total Cash: {total_cash}\n"
            f"Total Debt: {total_debt}\n"
            f"P/E Ratio: {pe_ratio}"
        )
    except Exception as e:
        data = f"Error fetching data: {str(e)}"
        
    logger.info(f"get_fundamentals returning: {data}")
    return data


def get_outlook(ticker: str, days: int = 30, sims: int = 5000) -> str:
    """
    Monte-Carlo experimental outlook using log-return simulation.

    Returns:
        A textual forecast containing:
        - probability of positive return
        - expected return
        - expected volatility
        - Sharpe ratio
        - qualitative classification
    """
    logger.info(f"get_outlook called for: {ticker}")

    try:
        # Download prices
        data = yf.download(ticker, period="3y", auto_adjust=True, progress=False)

        # Validate download
        if data is None or data.empty:
            raise ValueError(f"No price data found for {ticker}")

        close_prices = data["Close"].dropna()

        if close_prices.empty:
            raise ValueError("Close price series is empty")

        if len(close_prices) < 60:
            raise ValueError("Not enough historical data for simulation")

        # Daily log returns
        log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

        if log_returns.empty:
            raise ValueError("Log returns are empty")

        # Convert to SAFE SCALARS
        # mu = float(log_returns.mean())
        # sigma = float(log_returns.std())
        # last_price = float(close_prices.iloc[-1])
        mu = log_returns.mean().item()
        sigma = log_returns.std().item()
        last_price = close_prices.iloc[-1].item() if hasattr(close_prices.iloc[-1], "item") else float(close_prices.iloc[-1])


        if np.isnan(mu) or np.isnan(sigma):
            raise ValueError("mu or sigma is NaN (invalid)")

        if sigma == 0:
            raise ValueError("Standard deviation is zero (flat price series)")

        # Monte-Carlo simulation
        noise_matrix = np.random.normal(mu, sigma, size=(sims, days))
        price_paths = last_price * np.exp(noise_matrix.cumsum(axis=1))
        final_prices = price_paths[:, -1]

        final_returns = (final_prices - last_price) / last_price

        prob_up = float((final_returns > 0).mean())
        expected_return = float(final_returns.mean())
        volatility = float(final_returns.std())

        sharpe = float((mu / sigma) * np.sqrt(252)) if sigma > 0 else 0.0

        # Interpretation
        if prob_up > 0.65:
            outlook_label = "Bullish"
        elif prob_up > 0.55:
            outlook_label = "Moderately Bullish"
        elif prob_up > 0.45:
            outlook_label = "Neutral / Uncertain"
        else:
            outlook_label = "Bearish"

        summary = (
            f"Monte-Carlo Outlook for {ticker} ({days}-day horizon):\n"
            f"- Outlook: {outlook_label}\n"
            f"- Probability stock ends higher: {prob_up:.2%}\n"
            f"- Expected return: {expected_return:.2%}\n"
            f"- Expected volatility: {volatility:.2%}\n"
            f"- Sharpe ratio (est.): {sharpe:.2f}\n"
            f"*This forecast is experimental and based on Monte-Carlo simulation.*"
        )

    except Exception as e:
        summary = f"Error generating outlook: {str(e)}"

    logger.info(f"get_outlook returning: {summary}")
    return summary

def parse_sec_filing(ticker: str) -> str:
    """Parses the latest SEC filing for risks (Mock)."""
    logger.info(f"parse_sec_filing called for: {ticker}")
    # Mock data for MVP
    data = f"Latest 10-K for {ticker} highlights risks in: Supply chain disruptions, Antitrust litigation, and Foreign exchange fluctuations."
    logger.info(f"parse_sec_filing returning: {data}")
    return data

def score_sentiment(text: str) -> str:
    """Scores the sentiment of a given text (Mock)."""
    logger.info(f"score_sentiment called.")
    # Mock logic
    score = 75
    label = "Positive"
    return f"Sentiment Score: {score} ({label})"

In [38]:
from google.adk.agents import Agent, LlmAgent
from google.adk.tools import AgentTool, google_search

MODEL_NAME = "gemini-2.5-flash"

# 1. Quant Agent
quant_agent = Agent(
    name="QuantAgent",
    model=MODEL_NAME,
    description="A financial analyst that provides fundamental data.",
    instruction="You are a financial analyst. Use the `get_fundamentals` tool to analyze the company's financial health. Return a concise text summary of the fundamentals.",
    tools=[get_fundamentals]
)

# 2. Investigator Agent
investigator_agent = Agent(
    name="InvestigatorAgent",
    model=MODEL_NAME,
    description="A risk investigator that finds news, regulatory issues, and sentiment.",
    instruction="""You are a risk investigator. Use the `google_search` tool to identify risks, regulatory issues, and market sentiment.
    Search for terms like 'lawsuits', 'regulatory investigation', 'scandal', and 'analyst sentiment'.
    Use `score_sentiment` to evaluate the mood of recent headlines if needed.
    Return a concise text summary of the risks and sentiment found.""",
    tools=[score_sentiment]
)

# 3 Research Agent
research_agent = Agent(
    name = "ReasearchAgent",
    model = MODEL_NAME,
    description = "A research agent that finds new, headlines, issues.",
    instruction="""You are a reasearch agent thatuses 'google_search' tool to identify the latest news, regulatory issues, investigations against the firm, scandals, analyst sentiments, user sentiments from various sorces as shown but the 'google_search' agent.
    Focus on:
        - Lawsuits and regulatory investigations
        - Major product or strategy news
        - Analyst upgrades/downgrades

    Summarize the key risks and overall sentiment, with short inline citations like [1], [2].

    CRITICAL: Do not add information from your own, use the available online results from 'google_search' tool and then return a concise text summary.
    """,
    tools = [google_search]
)

# 4. Filings Agent
filings_agent = Agent(
    name="FilingsAgent",
    model=MODEL_NAME,
    description="A specialist in SEC filings.",
    instruction="""You are an expert in analyzing SEC filings (10-K, 10-Q). 
    Use the `parse_sec_filing` tool to extract key risks and financial highlights from the latest reports.
    Summarize the top 3 critical insights.""",
    tools=[parse_sec_filing]
)

# 5. Futurist Agent
futurist_agent = Agent(
    name="FuturistAgent",
    model=MODEL_NAME,
    description="A market futurist that provides an experimental quant forecast.",
    instruction="Use the `get_outlook` tool to produce a 30-day experimental forecast. "
                "Summarize the probability of positive return, expected return, "
                "volatility, and classification (bullish/bearish)."
                "CRITICAL: Wait for tool to finish genrating results and then move ahead.",
    tools=[get_outlook]
)

# quant_agent = AgentTool(agent=quant_agent)
# investigator_agent = AgentTool(agent=investigator_agent)
# filings_agent = AgentTool(agent=filings_agent)
# futurist_agent = AgentTool(agent=futurist_agent)

# 6. Manager Agent (The Orchestrator)
# Due to API limitations with nested agent calls, the Manager now uses tools directly.
class ManagerAgent(LlmAgent):
    def __init__(self):
        super().__init__(
            name="ManagerAgent",
            model=MODEL_NAME,
            instruction="""You are the Chief Financial Analyst of an investment firm.

            Your goal is to produce a comprehensive 3-paragraph report on a company
            based on input from your specialist agents.

            You can call these agent-tools:
            - QuantAgent: fetches and summarizes fundamentals using market data.
            - ResearchAgent: searches the web (via google_search) for news & risks.
            - FilingsAgent: extracts key risks & highlights from recent SEC filings.
            - FuturistAgent: provides an experimental short-term outlook.

            Process:
            1. Call QuantAgent to get the financial fundamentals.
            2. Call ResearchAgent to get recent news, risks, and sentiment.
            3. Call FilingsAgent to get SEC filing insights.
            4. Call FuturistAgent to get the outlook.
            5. Synthesize everything into EXACTLY three sections:

            - **Summary** (Fundamentals & Filings Highlights)
            - **Risks & Recent Developments** (News/Risks/Sentiment)
            - **Experimental Outlook** (Forecast; clearly marked as experimental, not advice)

            Each section should be 2-4 sentences. Keep the tone clear and professional.
    """,
            tools=[
               AgentTool(agent=quant_agent),
               AgentTool(agent=research_agent),
               AgentTool(agent=filings_agent),
               AgentTool(agent=futurist_agent),
            ]
        )

manager_agent = ManagerAgent()

In [40]:
# Initialize Runner
runner = InMemoryRunner(agent=manager_agent)

# Run the agent
await runner.run_debug("Analyze Nvidia (NVDA)")

INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False



 ### Created new session: debug_session_id

User > Analyze Nvidia (NVDA)


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:goo

ManagerAgent > Here is a comprehensive report on Nvidia (NVDA):

**Summary**
Nvidia (NVDA) demonstrates robust financial health, characterized by a substantial market capitalization exceeding $4.3 trillion and impressive revenue growth of 62.5%. The company maintains high operating margins of approximately 63.17% and a healthy cash position of $60.61 billion against $10.48 billion in total debt, with a P/E ratio of 44.28. However, recent SEC filings highlight critical risks including potential supply chain disruptions, exposure to antitrust litigation, and the impact of foreign exchange fluctuations on financial performance.

**Risks & Recent Developments**
Nvidia has recently reported record financial results, with data center revenue reaching an all-time high, driven by strong demand for its AI computing platforms and strategic partnerships with major tech companies. Analyst and user sentiment is overwhelmingly bullish, though some concerns exist regarding the company's high valuatio

[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Nvidia (NVDA) fundamentals'
         },
         id='adk-793e4e65-edc4-4032-b449-d761e00cf55a',
         name='QuantAgent'
       ),
       thought_signature=b'\n\xee\x05\x01\xd1\xed\x8aoz<\xd9\x01x\xa5mN7\xd0e\xf6Od\xcf\x84\xa2\x8c\xf2\x9f\xef\x1d>\x805\xa2m\x8bf\xb7\xb2~\x99>E2\xa7(\xb0\xad\xcbw\x95#-\xb9a\xf1\xf4\x9c\x0e*\xa2\x00\x89N\xfc\x81\xdd\x88\xf5\xcc;\xcb\xc6R\x1f\x89[/\xf9\x84tY&tp\xf5\x88W{\xb9g&\xbe\xa1\xe6\x9eQ...'
     ),
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Nvidia (NVDA) recent news, risks, and sentiment'
         },
         id='adk-bc7fd2f5-c5dc-4bdf-b928-56b0bb0bec8e',
         name='ReasearchAgent'
       )
     ),
     Part(
       function_call=FunctionCall(
         args={
           'request': 'Nvidia (NVDA) key risks and highlights from recent SEC filings'
     

In [ ]:
# def get_outlook(ticker: str, days: int = 30, sims: int = 5000) -> str:
#     """
#     Monte-Carlo experimental outlook using log-return simulation.

#     Returns:
#         A textual forecast containing:
#         - probability of positive return
#         - expected return
#         - expected volatility
#         - Sharpe ratio
#         - qualitative classification
#     """
#     logger.info(f"get_outlook_mc called for: {ticker}")

#     try:
#         # Download prices
#         data = yf.download(ticker, period="3y", auto_adjust=True, progress=False)

#         # Validate download
#         if data is None or data.empty:
#             raise ValueError(f"No price data found for {ticker}")

#         close_prices = data["Close"].dropna()

#         if close_prices.empty:
#             raise ValueError("Close price series is empty")

#         if len(close_prices) < 60:
#             raise ValueError("Not enough historical data for simulation")

#         # Daily log returns
#         log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

#         if log_returns.empty:
#             raise ValueError("Log returns are empty")

#         # Convert to SAFE SCALARS
#         # mu = float(log_returns.mean())
#         # sigma = float(log_returns.std())
#         # last_price = float(close_prices.iloc[-1])
#         mu = log_returns.mean().item()
#         sigma = log_returns.std().item()
#         last_price = close_prices.iloc[-1].item() if hasattr(close_prices.iloc[-1], "item") else float(close_prices.iloc[-1])


#         if np.isnan(mu) or np.isnan(sigma):
#             raise ValueError("mu or sigma is NaN (invalid)")

#         if sigma == 0:
#             raise ValueError("Standard deviation is zero (flat price series)")

#         # Monte-Carlo simulation
#         noise_matrix = np.random.normal(mu, sigma, size=(sims, days))
#         price_paths = last_price * np.exp(noise_matrix.cumsum(axis=1))
#         final_prices = price_paths[:, -1]

#         final_returns = (final_prices - last_price) / last_price

#         prob_up = float((final_returns > 0).mean())
#         expected_return = float(final_returns.mean())
#         volatility = float(final_returns.std())

#         sharpe = float((mu / sigma) * np.sqrt(252)) if sigma > 0 else 0.0

#         # Interpretation
#         if prob_up > 0.65:
#             outlook_label = "Bullish"
#         elif prob_up > 0.55:
#             outlook_label = "Moderately Bullish"
#         elif prob_up > 0.45:
#             outlook_label = "Neutral / Uncertain"
#         else:
#             outlook_label = "Bearish"

#         summary = (
#             f"Monte-Carlo Outlook for {ticker} ({days}-day horizon):\n"
#             f"- Outlook: {outlook_label}\n"
#             f"- Probability stock ends higher: {prob_up:.2%}\n"
#             f"- Expected return: {expected_return:.2%}\n"
#             f"- Expected volatility: {volatility:.2%}\n"
#             f"- Sharpe ratio (est.): {sharpe:.2f}\n"
#             f"*This forecast is experimental and based on Monte-Carlo simulation.*"
#         )

#     except Exception as e:
#         summary = f"Error generating outlook: {str(e)}"

#     logger.info(f"get_outlook_mc returning: {summary}")
#     return summary

In [33]:
res = get_outlook("TSLA")

INFO:__main__:get_outlook_mc called for: TSLA
INFO:__main__:get_outlook_mc returning: Monte-Carlo Outlook for TSLA (30-day horizon):
- Outlook: Moderately Bullish
- Probability stock ends higher: 55.56%
- Expected return: 5.22%
- Expected volatility: 22.43%
- Sharpe ratio (est.): 0.46
*This forecast is experimental and based on Monte-Carlo simulation.*


In [34]:
res = get_outlook("AAPL")

INFO:__main__:get_outlook_mc called for: AAPL
INFO:__main__:get_outlook_mc returning: Monte-Carlo Outlook for AAPL (30-day horizon):
- Outlook: Moderately Bullish
- Probability stock ends higher: 61.50%
- Expected return: 2.91%
- Expected volatility: 9.18%
- Sharpe ratio (est.): 0.78
*This forecast is experimental and based on Monte-Carlo simulation.*


In [35]:
res = get_outlook("NVDA")

INFO:__main__:get_outlook_mc called for: NVDA
INFO:__main__:get_outlook_mc returning: Monte-Carlo Outlook for NVDA (30-day horizon):
- Outlook: Bullish
- Probability stock ends higher: 71.74%
- Expected return: 12.30%
- Expected volatility: 20.00%
- Sharpe ratio (est.): 1.60
*This forecast is experimental and based on Monte-Carlo simulation.*
